
# Phase 7 — SHML System Integration

## Objective

Integrate the existing SHML components into a
reproducible end-to-end machine-learning workflow.

## Step 7.1 — Safety Controller Verification

Verify that the original Phase 5 safety controller
correctly accepts and rejects recovery candidates.

## Scope

- Dataset: UCI Default of Credit Card Clients
- Model: Logistic Regression
- Failure type: BILL_AMT1 missingness
- Recovery: Training-median imputation
- Safety: F1, recall, and data-integrity checks


In [11]:

# Phase 7.1 — Test original safety controller

import numpy as np
import pandas as pd

print("RUNNING PHASE 7.1 SAFETY TESTS")

healthy_metrics = {
    "accuracy": 0.8087,
    "precision": 0.6948,
    "recall": 0.2402,
    "f1": 0.3570,
    "roc_auc": 0.7130
}

test_cases = [
    ("Excessive F1 loss", 0.75, 1.00, True, "REJECT"),
    ("Excessive recall loss", 1.00, 0.75, True, "REJECT"),
    ("Data integrity failure", 1.00, 1.00, False, "REJECT"),
    ("Valid recovery", 0.95, 0.95, True, "ACCEPT")
]

results = []

for name, f1_factor, recall_factor, integrity, expected in test_cases:
    candidate = healthy_metrics.copy()
    candidate["f1"] *= f1_factor
    candidate["recall"] *= recall_factor

    outcome = evaluate_recovery_safety(
        recovered_metrics=candidate,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=integrity
    )

    results.append({
        "Test": name,
        "Expected": expected,
        "Actual": outcome["decision"],
        "Passed": outcome["decision"] == expected
    })

phase7_safety_results = pd.DataFrame(results)

print(phase7_safety_results.to_string(index=False))

assert phase7_safety_results["Passed"].all()

print("\nPhase 7.1 verification passed.")


RUNNING PHASE 7.1 SAFETY TESTS
                  Test Expected Actual  Passed
     Excessive F1 loss   REJECT REJECT    True
 Excessive recall loss   REJECT REJECT    True
Data integrity failure   REJECT REJECT    True
        Valid recovery   ACCEPT ACCEPT    True

Phase 7.1 verification passed.


In [12]:

# src/safety.py
# SHML recovery safety controller

import numpy as np


def evaluate_recovery_safety(
    recovered_metrics,
    healthy_metrics,
    data_integrity_passed,
    max_f1_loss=0.10,
    max_recall_loss=0.10
):
    """
    Evaluate offline recovery acceptance criteria.

    Returns:
        dict: Safety decision, reason, and performance losses.
    """

    required_metrics = [
        "accuracy", "precision", "recall", "f1", "roc_auc"
    ]

    metrics_valid = all(
        metric in recovered_metrics
        and np.isfinite(recovered_metrics[metric])
        and 0 <= recovered_metrics[metric] <= 1
        for metric in required_metrics
    )

    baseline_valid = (
        healthy_metrics["f1"] > 0
        and healthy_metrics["recall"] > 0
    )

    if not metrics_valid or not baseline_valid:
        return {
            "decision": "REJECT",
            "reason": "Invalid evaluation metrics or baseline",
            "f1_loss_pct": np.nan,
            "recall_loss_pct": np.nan
        }

    f1_loss = (
        healthy_metrics["f1"] - recovered_metrics["f1"]
    ) / healthy_metrics["f1"]

    recall_loss = (
        healthy_metrics["recall"] - recovered_metrics["recall"]
    ) / healthy_metrics["recall"]

    reasons = []

    if not data_integrity_passed:
        reasons.append("Data integrity check failed")

    if f1_loss > max_f1_loss:
        reasons.append("F1 loss exceeds tolerance")

    if recall_loss > max_recall_loss:
        reasons.append("Recall loss exceeds tolerance")

    decision = "REJECT" if reasons else "ACCEPT"

    return {
        "decision": decision,
        "reason": "; ".join(reasons) if reasons else "All checks passed",
        "f1_loss_pct": f1_loss * 100,
        "recall_loss_pct": recall_loss * 100
    }


In [14]:

# Phase 7.2 — Verify reusable safety module
from src.safety import evaluate_recovery_safety as module_safety

test_result = module_safety(
    recovered_metrics={
        "accuracy": 0.8087,
        "precision": 0.6948,
        "recall": 0.2402,
        "f1": 0.3570,
        "roc_auc": 0.7130
    },
    healthy_metrics={
        "f1": 0.3570,
        "recall": 0.2402
    },
    data_integrity_passed=True
)

print("Decision:", test_result["decision"])
print("Reason:", test_result["reason"])

assert test_result["decision"] == "ACCEPT"

print("Phase 7.2 safety module import passed.")


Decision: ACCEPT
Reason: All checks passed
Phase 7.2 safety module import passed.


In [17]:

# Phase 7.2 — Verify detection module

import pandas as pd
from src.detection import detect_feature_missingness

test_batch = pd.DataFrame({
    "BILL_AMT1": [100, 200, None, None, 500]
})

result = detect_feature_missingness(
    X_batch=test_batch,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05
)

print("Current missingness:", result["current_missingness"])
print("Missingness increase:", result["missingness_increase"])
print("Failure detected:", result["failure_detected"])

assert result["failure_detected"] is True

print("Phase 7.2 detection module verification passed.")


Current missingness: 0.4
Missingness increase: 0.4
Failure detected: True
Phase 7.2 detection module verification passed.


In [19]:


# Phase 7.2 — Verify diagnosis module
from src.diagnosis import diagnose_failure

test_detection = {
    "feature": "BILL_AMT1",
    "current_missingness": 0.15,
    "failure_detected": True
}

diagnosis_result = diagnose_failure(test_detection)

print("Feature:", diagnosis_result["feature"])
print("Failure type:", diagnosis_result["failure_type"])
print("Severity:", diagnosis_result["severity"])
print("Recovery action:", diagnosis_result["recovery_action"])
print("Explanation:", diagnosis_result["explanation"])

assert diagnosis_result["severity"] == "Moderate"
assert diagnosis_result["recovery_action"] == "Median imputation"

print("\nPhase 7.2 diagnosis module verification passed.")


Feature: BILL_AMT1
Failure type: Numerical missingness
Severity: Moderate
Recovery action: Median imputation
Explanation: BILL_AMT1 has 15.00% missing values. Severity: Moderate. Recommended action: Median imputation.

Phase 7.2 diagnosis module verification passed.


In [21]:

# Phase 7.2 — Verify recovery module
from src.recovery import recover_missingness

test_batch = pd.DataFrame({
    "BILL_AMT1": [1000, None, 3000, None, 5000],
    "AGE": [25, 30, 35, 40, 45]
})

test_diagnosis = {
    "feature": "BILL_AMT1",
    "severity": "Moderate",
    "recovery_action": "Median imputation"
}

# Training median from the original SHML experiment
training_median = 22225.5

recovered_batch, recovery_log = recover_missingness(
    X_batch=test_batch,
    diagnosis=test_diagnosis,
    training_median=training_median
)

print("Original BILL_AMT1:")
print(test_batch["BILL_AMT1"].tolist())

print("\nRecovered BILL_AMT1:")
print(recovered_batch["BILL_AMT1"].tolist())

print("\nRecovery log:")
print(recovery_log)

assert recovery_log["missing_before"] == 2
assert recovery_log["missing_after"] == 0
assert recovery_log["values_imputed"] == 2

# Verify that original data was not modified
assert test_batch["BILL_AMT1"].isna().sum() == 2

# Verify unrelated feature remains unchanged
assert recovered_batch["AGE"].equals(test_batch["AGE"])

print("\nPhase 7.2 recovery module verification passed.")


Original BILL_AMT1:
[1000.0, nan, 3000.0, nan, 5000.0]

Recovered BILL_AMT1:
[1000.0, 22225.5, 3000.0, 22225.5, 5000.0]

Recovery log:
{'feature': 'BILL_AMT1', 'severity': 'Moderate', 'action': 'Median imputation', 'missing_before': 2, 'missing_after': 0, 'values_imputed': 2, 'recovery_applied': True}

Phase 7.2 recovery module verification passed.


In [22]:

# Phase 7.3 — Import all SHML components

from src.detection import detect_feature_missingness
from src.diagnosis import diagnose_failure
from src.recovery import recover_missingness
from src.safety import evaluate_recovery_safety

print("All four SHML modules imported successfully.")


All four SHML modules imported successfully.


In [23]:

# Phase 7.3.2 — Integrate detection, diagnosis, recovery

import pandas as pd

# Small test batch with 20% missingness
X_test_batch = pd.DataFrame({
    "BILL_AMT1": [1000, 2000, None, 4000, 5000],
    "AGE": [25, 30, 35, 40, 45]
})

# 1. Detect
detection_result = detect_feature_missingness(
    X_batch=X_test_batch,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05
)

# 2. Diagnose
diagnosis_result = diagnose_failure(detection_result)

# 3. Recover
X_recovered, recovery_log = recover_missingness(
    X_batch=X_test_batch,
    diagnosis=diagnosis_result,
    training_median=22225.5
)

print("Detection:", detection_result["failure_detected"])
print("Severity:", diagnosis_result["severity"])
print("Recovery action:", diagnosis_result["recovery_action"])
print("Missing before:", recovery_log["missing_before"])
print("Missing after:", recovery_log["missing_after"])

assert detection_result["failure_detected"] is True
assert diagnosis_result["severity"] == "Moderate"
assert recovery_log["missing_after"] == 0
assert X_test_batch["BILL_AMT1"].isna().sum() == 1

print("\nPhase 7.3.2 integration test passed.")


Detection: True
Severity: Moderate
Recovery action: Median imputation
Missing before: 1
Missing after: 0

Phase 7.3.2 integration test passed.


In [24]:
from src.evaluation import evaluate_model

print("Phase 7.3.3 evaluation module imported successfully.")

Phase 7.3.3 evaluation module imported successfully.


In [25]:

# Phase 7.3.4 — Load dataset and reproduce data splits

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Locate the original dataset
data_dir = Path("data/working")

candidates = sorted(
    data_dir.glob("*credit*card*clients*.xls*")
)

if not candidates:
    raise FileNotFoundError(
        f"No dataset found in {data_dir}"
    )

DATA_PATH = candidates[0]

# Load original dataset
df = pd.read_excel(DATA_PATH, header=1)

target_column = "default payment next month"

X = df.drop(columns=["ID", target_column])
y = df[target_column]

# Reproduce Phase 5 train/validation/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Dataset:", DATA_PATH)
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

assert X_train.shape == (21000, 23)
assert X_val.shape == (4500, 23)
assert X_test.shape == (4500, 23)

print("\nPhase 7.3.4 dataset verification passed.")


Dataset: data/working/default of credit card clients.xls
Training: (21000, 23)
Validation: (4500, 23)
Testing: (4500, 23)

Phase 7.3.4 dataset verification passed.


In [26]:

# Phase 7.3.5 — Reproduce healthy baseline

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.evaluation import evaluate_model

# 1. Fit scaler on training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# 2. Train the original Logistic Regression model
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

baseline_model.fit(X_train_scaled, y_train)

# 3. Evaluate on unchanged validation data
healthy_metrics = evaluate_model(
    X_batch=X_val,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

# 4. Display results
print("Phase 7 — Healthy Baseline Metrics")

for metric, value in healthy_metrics.items():
    print(f"{metric}: {value:.4f}")

# 5. Verify consistency with Phase 5
assert abs(healthy_metrics["f1"] - 0.3570) < 0.0001

print("\nPhase 7.3.5 baseline reproduction passed.")


Phase 7 — Healthy Baseline Metrics
accuracy: 0.8087
precision: 0.6948
recall: 0.2402
f1: 0.3570
roc_auc: 0.7130

Phase 7.3.5 baseline reproduction passed.


In [27]:

# Phase 7.3.6 — Reproduce missingness scenarios

import numpy as np

missingness_levels = [0.00, 0.05, 0.15, 0.30]

corrupted_batches = {}

for rate in missingness_levels:

    # Preserve the original validation dataset
    X_corrupted = X_val.copy(deep=True)

    # Calculate number of values to corrupt
    n_missing = int(len(X_corrupted) * rate)

    # Use the original Phase 5 random seed
    rng = np.random.default_rng(RANDOM_STATE)

    missing_positions = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    # Inject missingness only into BILL_AMT1
    feature_idx = X_corrupted.columns.get_loc("BILL_AMT1")

    X_corrupted.iloc[
        missing_positions, feature_idx
    ] = np.nan

    corrupted_batches[rate] = X_corrupted

    print(
        f"Missingness: {rate:.0%} | "
        f"Missing values: "
        f"{X_corrupted['BILL_AMT1'].isna().sum()}"
    )

# Verify original validation data remains unchanged
assert X_val["BILL_AMT1"].isna().sum() == 0

# Verify expected corruption counts
expected_missing = {
    0.00: 0,
    0.05: 225,
    0.15: 675,
    0.30: 1350
}

for rate in missingness_levels:
    assert (
        corrupted_batches[rate]["BILL_AMT1"].isna().sum()
        == expected_missing[rate]
    )

print("\nPhase 7.3.6 missingness injection verification passed.")


Missingness: 0% | Missing values: 0
Missingness: 5% | Missing values: 225
Missingness: 15% | Missing values: 675
Missingness: 30% | Missing values: 1350

Phase 7.3.6 missingness injection verification passed.


In [28]:

# Phase 7.3.7 — Integrated SHML pipeline

import pandas as pd

training_median = X_train["BILL_AMT1"].median()

pipeline_results = []

for rate in missingness_levels:

    # 1. Retrieve corrupted validation batch
    X_batch = corrupted_batches[rate]

    # 2. Detect missingness failure
    detection = detect_feature_missingness(
        X_batch=X_batch,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    # 3. Diagnose the failure
    diagnosis = diagnose_failure(detection)

    # 4. Apply the recommended recovery
    X_recovered, recovery_log = recover_missingness(
        X_batch=X_batch,
        diagnosis=diagnosis,
        training_median=training_median
    )

    # 5. Verify recovered data integrity
    integrity_passed = (
        X_recovered.isna().sum().sum() == 0
        and X_recovered.shape == X_batch.shape
        and X_recovered.columns.equals(X_batch.columns)
    )

    # 6. Evaluate using the unchanged baseline model
    recovered_metrics = evaluate_model(
        X_batch=X_recovered,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 7. Apply the original safety controller
    safety = evaluate_recovery_safety(
        recovered_metrics=recovered_metrics,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=integrity_passed,
        max_f1_loss=0.10,
        max_recall_loss=0.10
    )

    # 8. Record experiment results
    pipeline_results.append({
        "missingness": rate,
        "detected": detection["failure_detected"],
        "severity": diagnosis["severity"],
        "action": diagnosis["recovery_action"],
        "values_imputed": recovery_log["values_imputed"],
        "integrity_passed": integrity_passed,
        "f1": recovered_metrics["f1"],
        "recall": recovered_metrics["recall"],
        "roc_auc": recovered_metrics["roc_auc"],
        "decision": safety["decision"],
        "reason": safety["reason"]
    })

pipeline_df = pd.DataFrame(pipeline_results)

display(pipeline_df.round(4))

assert len(pipeline_df) == 4
assert pipeline_df["integrity_passed"].all()
assert pipeline_df["decision"].isin(["ACCEPT", "REJECT"]).all()

print("\nPhase 7.3.7 integrated pipeline verification passed.")


,missingness,detected,severity,action,values_imputed,integrity_passed,f1,recall,roc_auc,decision,reason
0,0.00,False,None,No correction,0,True,0.3570,0.2402,0.7130,ACCEPT,All checks passed
1,0.05,False,Below alert threshold,Routine median preprocessing,225,True,0.3589,0.2422,0.7132,ACCEPT,All checks passed
2,0.15,True,Moderate,Median imputation,675,True,0.3609,0.2452,0.7090,ACCEPT,All checks passed
3,0.30,True,Severe,Median imputation,1350,True,0.3737,0.2573,0.7111,ACCEPT,All checks passed



Phase 7.3.7 integrated pipeline verification passed.


In [29]:

# Phase 7.3.8 — Verify integrated pipeline results

expected_detection = {
    0.00: False,
    0.05: False,
    0.15: True,
    0.30: True
}

expected_f1 = {
    0.00: 0.356983,
    0.05: 0.358898,
    0.15: 0.360947,
    0.30: 0.373723
}

for _, row in pipeline_df.iterrows():
    rate = float(row["missingness"])

    assert bool(row["detected"]) == expected_detection[rate], (
        f"Detection mismatch at {rate:.0%}"
    )

    assert abs(row["f1"] - expected_f1[rate]) < 0.0001, (
        f"F1 mismatch at {rate:.0%}"
    )

    assert row["decision"] == "ACCEPT", (
        f"Unexpected safety decision at {rate:.0%}"
    )

print("All detection decisions match Phase 5.")
print("All F1-scores match Phase 5 within tolerance.")
print("All four safety decisions are ACCEPT.")
print("\nPhase 7.3.8 reproducibility verification passed.")


All detection decisions match Phase 5.
All F1-scores match Phase 5 within tolerance.
All four safety decisions are ACCEPT.

Phase 7.3.8 reproducibility verification passed.


In [30]:

# Phase 7.4.1 — Verify unsafe recovery rejection

# Create a candidate with a 25% relative F1 loss
unsafe_metrics = healthy_metrics.copy()

unsafe_metrics["f1"] = healthy_metrics["f1"] * 0.75

# Evaluate the unsafe candidate
unsafe_decision = evaluate_recovery_safety(
    recovered_metrics=unsafe_metrics,
    healthy_metrics=healthy_metrics,
    data_integrity_passed=True,
    max_f1_loss=0.10,
    max_recall_loss=0.10
)

print("Safety decision:", unsafe_decision["decision"])
print("Reason:", unsafe_decision["reason"])
print("F1 loss (%):", round(unsafe_decision["f1_loss_pct"], 2))

assert unsafe_decision["decision"] == "REJECT"
assert abs(unsafe_decision["f1_loss_pct"] - 25.0) < 0.0001

print("\nPhase 7.4.1 unsafe recovery rejection passed.")


Safety decision: REJECT
Reason: F1 loss exceeds tolerance
F1 loss (%): 25.0

Phase 7.4.1 unsafe recovery rejection passed.


In [31]:

# Phase 7.4.2 — Test rollback and state preservation

# Simulated accepted model state
active_state = {
    "version": "baseline_v1",
    "status": "ACTIVE"
}

# Simulated candidate state
candidate_state = {
    "version": "candidate_unsafe_v2",
    "status": "PENDING"
}

# Apply the safety decision from Phase 7.4.1
if unsafe_decision["decision"] == "ACCEPT":
    active_state = {
        "version": candidate_state["version"],
        "status": "ACTIVE"
    }
    transition = "PROMOTED"
else:
    transition = "REJECTED — ACTIVE STATE PRESERVED"

print("Candidate:", candidate_state["version"])
print("Safety decision:", unsafe_decision["decision"])
print("Transition:", transition)
print("Active version:", active_state["version"])

# Verify rejected candidate was not promoted
assert unsafe_decision["decision"] == "REJECT"
assert active_state["version"] == "baseline_v1"
assert active_state["status"] == "ACTIVE"

print("\nPhase 7.4.2 state preservation verification passed.")


Candidate: candidate_unsafe_v2
Safety decision: REJECT
Transition: REJECTED — ACTIVE STATE PRESERVED
Active version: baseline_v1

Phase 7.4.2 state preservation verification passed.


In [32]:

# Phase 7.4.3 — Test successful candidate promotion

# Preserve the original active state
active_state = {
    "version": "baseline_v1",
    "status": "ACTIVE"
}

# Create a safe candidate with 5% relative F1 and recall losses
safe_metrics = healthy_metrics.copy()
safe_metrics["f1"] *= 0.95
safe_metrics["recall"] *= 0.95

# Evaluate the candidate
safe_decision = evaluate_recovery_safety(
    recovered_metrics=safe_metrics,
    healthy_metrics=healthy_metrics,
    data_integrity_passed=True,
    max_f1_loss=0.10,
    max_recall_loss=0.10
)

candidate_state = {
    "version": "candidate_safe_v2",
    "status": "PENDING"
}

# Apply safety decision to simulated state
if safe_decision["decision"] == "ACCEPT":
    active_state = {
        "version": candidate_state["version"],
        "status": "ACTIVE"
    }
    transition = "PROMOTED"
else:
    transition = "REJECTED — ACTIVE STATE PRESERVED"

print("Candidate:", candidate_state["version"])
print("Safety decision:", safe_decision["decision"])
print("Transition:", transition)
print("Active version:", active_state["version"])

assert safe_decision["decision"] == "ACCEPT"
assert active_state["version"] == "candidate_safe_v2"
assert active_state["status"] == "ACTIVE"

print("\nPhase 7.4.3 successful promotion verification passed.")


Candidate: candidate_safe_v2
Safety decision: ACCEPT
Transition: PROMOTED
Active version: candidate_safe_v2

Phase 7.4.3 successful promotion verification passed.


In [33]:

# Phase 7.4.4 — Test invalid evaluation metrics

import numpy as np

invalid_cases = [
    ("NaN F1", "f1", np.nan),
    ("F1 above 1", "f1", 1.5),
    ("Negative recall", "recall", -0.1),
    ("Invalid ROC-AUC", "roc_auc", 1.2)
]

invalid_results = []

for name, metric, invalid_value in invalid_cases:

    candidate_metrics = healthy_metrics.copy()
    candidate_metrics[metric] = invalid_value

    result = evaluate_recovery_safety(
        recovered_metrics=candidate_metrics,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=True
    )

    invalid_results.append({
        "Test": name,
        "Decision": result["decision"],
        "Passed": result["decision"] == "REJECT"
    })

invalid_df = pd.DataFrame(invalid_results)

display(invalid_df)

assert invalid_df["Passed"].all()

print("\nPhase 7.4.4 invalid metrics verification passed.")


,Test,Decision,Passed
0,NaN F1,REJECT,True
1,F1 above 1,REJECT,True
2,Negative recall,REJECT,True
3,Invalid ROC-AUC,REJECT,True



Phase 7.4.4 invalid metrics verification passed.


In [34]:

# Phase 7.4.5 — Test invalid healthy baselines

import numpy as np
import pandas as pd

baseline_cases = [
    ("Zero baseline F1", "f1", 0.0),
    ("NaN baseline F1", "f1", np.nan),
    ("Infinite baseline recall", "recall", np.inf),
    ("Baseline F1 above 1", "f1", 1.5)
]

baseline_test_results = []

for name, metric, invalid_value in baseline_cases:

    invalid_baseline = healthy_metrics.copy()
    invalid_baseline[metric] = invalid_value

    try:
        result = evaluate_recovery_safety(
            recovered_metrics=healthy_metrics.copy(),
            healthy_metrics=invalid_baseline,
            data_integrity_passed=True
        )

        actual = result["decision"]

    except Exception as error:
        actual = f"ERROR: {type(error).__name__}"

    baseline_test_results.append({
        "Test": name,
        "Actual": actual,
        "Expected": "REJECT",
        "Passed": actual == "REJECT"
    })

baseline_test_df = pd.DataFrame(baseline_test_results)

display(baseline_test_df)

print(
    "Tests passed:",
    baseline_test_df["Passed"].sum(),
    "of",
    len(baseline_test_df)
)


,Test,Actual,Expected,Passed
0,Zero baseline F1,REJECT,REJECT,True
1,NaN baseline F1,REJECT,REJECT,True
2,Infinite baseline recall,ACCEPT,REJECT,False
3,Baseline F1 above 1,REJECT,REJECT,True


Tests passed: 3 of 4


In [35]:

# Phase 7.4.6 — Reload corrected safety module

import importlib
import src.safety

importlib.reload(src.safety)

evaluate_recovery_safety = (
    src.safety.evaluate_recovery_safety
)

print("Updated safety controller loaded.")


Updated safety controller loaded.


In [36]:

# Check which safety controller the notebook is using

import inspect
import importlib
import src.safety

importlib.reload(src.safety)

evaluate_recovery_safety = src.safety.evaluate_recovery_safety

print("Safety module location:")
print(src.safety.__file__)

print("\nCurrent baseline validation code:")
print(inspect.getsource(evaluate_recovery_safety))


Safety module location:
/Users/divya/Desktop/School/IU/GitStuff/SHML/src/safety.py

Current baseline validation code:
def evaluate_recovery_safety(
    recovered_metrics,
    healthy_metrics,
    data_integrity_passed,
    max_f1_loss=0.10,
    max_recall_loss=0.10
):
    """
    Evaluate offline recovery acceptance criteria.
    """

    required_metrics = [
        "accuracy", "precision", "recall", "f1", "roc_auc"
    ]

    metrics_valid = all(
        metric in recovered_metrics
        and np.isfinite(recovered_metrics[metric])
        and 0 <= recovered_metrics[metric] <= 1
        for metric in required_metrics
    )

    baseline_valid = all(
        metric in healthy_metrics
        and np.isfinite(healthy_metrics[metric])
        and 0 < healthy_metrics[metric] <= 1
        for metric in ["f1", "recall"]
    )

    if not metrics_valid or not baseline_valid:
        return {
            "decision": "REJECT",
            "reason": "Invalid evaluation metrics or baseline",
   

In [37]:

# Phase 7.4.7 — Retest corrected baseline validation

import numpy as np

test_cases = [
    ("Zero baseline F1", "f1", 0.0),
    ("NaN baseline F1", "f1", np.nan),
    ("Infinite baseline recall", "recall", np.inf),
    ("Baseline F1 above 1", "f1", 1.5)
]

passed = 0

for name, metric, value in test_cases:
    invalid_baseline = healthy_metrics.copy()
    invalid_baseline[metric] = value

    result = evaluate_recovery_safety(
        recovered_metrics=healthy_metrics.copy(),
        healthy_metrics=invalid_baseline,
        data_integrity_passed=True
    )

    success = result["decision"] == "REJECT"
    passed += success

    print(f"{name}: {result['decision']} | Passed: {success}")

print(f"\nTests passed: {passed} of {len(test_cases)}")

assert passed == len(test_cases)

print("Phase 7.4.7 baseline validation passed.")


Zero baseline F1: REJECT | Passed: True
NaN baseline F1: REJECT | Passed: True
Infinite baseline recall: REJECT | Passed: True
Baseline F1 above 1: REJECT | Passed: True

Tests passed: 4 of 4
Phase 7.4.7 baseline validation passed.


In [38]:

# Phase 7.4.8 — Verify recovery preserves original data

import pandas as pd

# Use the 30% missingness scenario
X_corrupted = corrupted_batches[0.30]

detection = detect_feature_missingness(
    X_batch=X_corrupted,
    feature="BILL_AMT1",
    reference_missingness=0.0,
    threshold=0.05
)

diagnosis = diagnose_failure(detection)

X_recovered, recovery_log = recover_missingness(
    X_batch=X_corrupted,
    diagnosis=diagnosis,
    training_median=training_median
)

# 1. Verify all other features remain unchanged
other_features = X_corrupted.columns.drop("BILL_AMT1")

other_features_preserved = X_recovered[
    other_features
].equals(X_corrupted[other_features])

# 2. Verify existing BILL_AMT1 values remain unchanged
observed_mask = X_corrupted["BILL_AMT1"].notna()

observed_values_preserved = X_recovered.loc[
    observed_mask, "BILL_AMT1"
].equals(
    X_corrupted.loc[observed_mask, "BILL_AMT1"]
)

# 3. Verify missing values were replaced by training median
missing_mask = X_corrupted["BILL_AMT1"].isna()

missing_values_corrected = (
    X_recovered.loc[missing_mask, "BILL_AMT1"]
    == training_median
).all()

# 4. Verify original corrupted batch was not modified
original_batch_preserved = X_corrupted.loc[
    missing_mask, "BILL_AMT1"
].isna().all()

print("Other features preserved:", other_features_preserved)
print("Observed values preserved:", observed_values_preserved)
print("Missing values corrected:", missing_values_corrected)
print("Original batch preserved:", original_batch_preserved)
print("Values imputed:", recovery_log["values_imputed"])

assert other_features_preserved
assert observed_values_preserved
assert missing_values_corrected
assert original_batch_preserved
assert recovery_log["values_imputed"] == 1350

print("\nPhase 7.4.8 data integrity verification passed.")


Other features preserved: True
Observed values preserved: True
Missing values corrected: True
Original batch preserved: True
Values imputed: 1350

Phase 7.4.8 data integrity verification passed.


In [39]:

# Phase 7.4.9 — Final integrated pipeline regression test

# Reuse the integrated pipeline from Phase 7.3.7
# with the updated safety controller.

expected_detection = {
    0.00: False,
    0.05: False,
    0.15: True,
    0.30: True
}

expected_f1 = {
    0.00: 0.356983,
    0.05: 0.358898,
    0.15: 0.360947,
    0.30: 0.373723
}

final_results = []

for rate in missingness_levels:
    X_batch = corrupted_batches[rate]

    detection = detect_feature_missingness(
        X_batch, "BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    diagnosis = diagnose_failure(detection)

    X_recovered, recovery_log = recover_missingness(
        X_batch, diagnosis, training_median
    )

    integrity_passed = (
        X_recovered.isna().sum().sum() == 0
        and X_recovered.shape == X_batch.shape
        and X_recovered.columns.equals(X_batch.columns)
    )

    metrics = evaluate_model(
        X_recovered, y_val, baseline_model, scaler
    )

    safety = evaluate_recovery_safety(
        recovered_metrics=metrics,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=integrity_passed
    )

    assert detection["failure_detected"] == expected_detection[rate]
    assert abs(metrics["f1"] - expected_f1[rate]) < 0.0001
    assert safety["decision"] == "ACCEPT"
    assert integrity_passed

    final_results.append({
        "Missingness": f"{rate:.0%}",
        "Detected": detection["failure_detected"],
        "F1": round(metrics["f1"], 6),
        "Decision": safety["decision"]
    })

display(pd.DataFrame(final_results))

print("\nPhase 7.4.9 final regression test passed.")


,Missingness,Detected,F1,Decision
0,0%,False,0.356983,ACCEPT
1,5%,False,0.358898,ACCEPT
2,15%,True,0.360947,ACCEPT
3,30%,True,0.373723,ACCEPT



Phase 7.4.9 final regression test passed.


In [42]:

    # Phase 7.5.3.3 — Train and evaluate healthy baseline

    # Fit scaler using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

    # Train Logistic Regression
baseline_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

baseline_model.fit(X_train_scaled, y_train)

    # Evaluate on untouched validation data
healthy_metrics = evaluate_model(
        X_batch=X_val,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

print("\nHealthy baseline metrics:")

for metric, value in healthy_metrics.items():
        print(f"{metric}: {value:.4f}")

    # Verify original Phase 3 baseline
assert abs(healthy_metrics["f1"] - 0.3570) < 0.0001
assert abs(healthy_metrics["recall"] - 0.2402) < 0.0001
assert abs(healthy_metrics["roc_auc"] - 0.7130) < 0.0001
assert abs(healthy_metrics["accuracy"] - 0.8087) < 0.0001

print("\nPhase 7.5.3.3 baseline verification passed.")



Healthy baseline metrics:
accuracy: 0.8087
precision: 0.6948
recall: 0.2402
f1: 0.3570
roc_auc: 0.7130

Phase 7.5.3.3 baseline verification passed.


In [43]:

# Phase 7.5.3.4 — Generate controlled missingness

missingness_levels = [0.00, 0.05, 0.15, 0.30]
corrupted_batches = {}

for rate in missingness_levels:

    # Create an independent copy of validation data
    X_corrupted = X_val.copy(deep=True)

    # Use the same random seed for each scenario
    rng = np.random.default_rng(42)

    n_missing = int(len(X_corrupted) * rate)

    missing_indices = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    # Inject missing values into BILL_AMT1 only
    X_corrupted.iloc[
        missing_indices,
        X_corrupted.columns.get_loc("BILL_AMT1")
    ] = np.nan

    corrupted_batches[rate] = X_corrupted

    actual_missing = X_corrupted["BILL_AMT1"].isna().sum()

    print(
        f"Missingness: {rate:.0%} | "
        f"Missing values: {actual_missing}"
    )

    assert actual_missing == n_missing

# Confirm original validation data remains unchanged
assert X_val["BILL_AMT1"].isna().sum() == 0

print("\nPhase 7.5.3.4 missingness verification passed.")


Missingness: 0% | Missing values: 0
Missingness: 5% | Missing values: 225
Missingness: 15% | Missing values: 675
Missingness: 30% | Missing values: 1350

Phase 7.5.3.4 missingness verification passed.
